#**Tugas Praktikum: Fine-tuning BERT/IndoBERT untuk Klasifikasi Teks Sederhana**

###**Nama      : Jousadrah Ramba**

###**Nim     : IK2311021**

#####Dataset yang digunakan adalah kumpulan teks ulasan produk dalam bahasa Indonesia. Dataset ini dirancang untuk membedakan sentimen pengguna (positif dan negatif) terhadap suatu layanan atau barang. Teks yang digunakan mencerminkan bahasa kasual sehari-hari yang sering ditemukan di platform ulasan daring.

#####  **Jumlah Data dan Label**
Total Seluruh Data: 60 sampel ulasan.

Distribusi Kategori Label:

Label 1 (Positif): 30 data (Contoh: "Saya sangat senang dengan layanan ini").

Label 0 (Negatif): 30 data (Contoh: "Saya sangat kecewa dengan produk ini").


 Setelah itu, data diacak dan dibagi menggunakan metode train_test_split dengan rasio 80% untuk data latih (48 data) dan 20% untuk data uji/evaluasi (12 data). Terakhir, data diubah ke format khusus yang dikenali oleh Hugging Face.

In [5]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Menyiapkan 60 data ulasan terdistribusi seimbang
data = {
    'text': [
        # === 30 DATA POSITIF (Label: 1) ===
        "Saya sangat senang dengan layanan ini", "Produk ini sangat bagus",
        "Pengiriman sangat cepat dan aman", "Saya puas sekali",
        "Kualitas barang luar biasa", "Pelayanannya ramah sekali",
        "Sangat direkomendasikan", "Barang sampai dengan kondisi baik",
        "Saya suka desainnya", "Sangat membantu pekerjaan saya",
        "Harga terjangkau dengan kualitas juara", "Respon penjual cepat dan solutif",
        "Packing sangat rapi dan aman", "Barang original sesuai deskripsi",
        "Fungsinya berjalan dengan sangat baik", "Aplikasi ini sangat mudah digunakan",
        "Sangat puas belanja di toko ini", "Material produk kokoh dan premium",
        "Ukuran pas banget sesuai pesanan", "Bakal langganan terus di sini",
        "Terima kasih, produknya memuaskan", "Pelayanan yang sangat profesional",
        "Proses transaksi cepat tanpa kendala", "Diskonnya gede dan barangnya bagus",
        "Kemasannya sangat rapi dan estetik", "Warnanya sangat cantik sesuai foto",
        "Sangat berguna untuk kebutuhan sehari-hari", "Rekomendasi banget untuk yang cari kualitas",
        "Kurirnya ramah dan sopan saat antar barang", "Kualitas di atas rata-rata untuk harga segini",

        # === 30 DATA NEGATIF (Label: 0) ===
        "Saya sangat kecewa dengan produk ini", "Barang rusak saat diterima",
        "Pengiriman sangat lama", "Pelayanan sangat buruk",
        "Tidak sesuai dengan deskripsi", "Sangat mengecewakan",
        "Jangan beli di sini", "Kualitas sangat buruk",
        "Saya menyesal membeli ini", "Sangat tidak direkomendasikan",
        "Ukuran tidak sesuai dengan yang dipesan", "Respon penjual sangat lambat",
        "Kemasan hancur dan robek saat sampai", "Barangnya cepat rusak baru dipakai sehari",
        "Bahan tipis sekali tidak sesuai harga", "Sangat menyesal belanja di toko ini",
        "Produk cacat dan tidak bisa digunakan", "Aplikasi sering eror dan lambat",
        "Pelayanannya tidak ramah sama sekali", "Kapok belanja di sini, tidak amanah",
        "Harga mahal tapi kualitas abal-abal", "Warna tidak sesuai dengan gambar",
        "Proses pengembalian barang ribet banget", "Barang tidak dikirim lengkap, kurang komponen",
        "Sangat tidak puas dengan hasil akhirnya", "Deskripsi produk menipu pembeli",
        "Kurirnya kasar dan tidak sopan", "Baterai cepat habis dan bocor",
        "Fungsinya tidak jalan sama sekali", "Tidak akan beli di toko ini lagi"
    ],
    'label': [1]*30 + [0]*30
}

df = pd.DataFrame(data)

# 2. Membagi data menjadi Latih (80%) dan Uji (20%)
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Konversi format data agar siap dipakai oleh model NLP
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

print(f"Dataset berhasil dimuat!")
print(f"Jumlah Data Latih: {len(train_df)} sampel | Jumlah Data Uji: {len(eval_df)} sampel")

Dataset berhasil dimuat!
Jumlah Data Latih: 48 sampel | Jumlah Data Uji: 12 sampel


#####**Tahapan Preprocessing/Tokenization**

#####Menginisialisasi komponen AutoTokenizer milik Hugging Face untuk mengonversi teks ulasan menjadi Token ID numerik. Fungsi di dalam sel ini secara otomatis menerapkan standar panjang input 128 token melalui mekanisme padding dan truncation bawaan pustaka Hugging Face agar data siap dilatih oleh model IndoBERT.

In [6]:
from transformers import AutoTokenizer

# Memanggil komponen tokenisasi resmi dari IndoBERT base-p2
model_checkpoint = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Fungsi untuk memproses teks mentah menjadi token angka
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Menerapkan tokenisasi ke seluruh data latih dan data uji
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

print("Proses Preprocessing dan Tokenisasi Selesai!")

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Proses Preprocessing dan Tokenisasi Selesai!


#####**Memuat Model IndoBERT**

#####Kita memanggil arsitektur utama model kecerdasan buatan IndoBERT base-p2. Karena model asli bawaannya ditujukan untuk tugas umum, kita secara spesifik menginstruksikan model untuk mengubah strukturnya agar mengenali 2 label saja (num_labels=2), yaitu sentimen positif atau negatif.

In [7]:
from transformers import AutoModelForSequenceClassification

# Memuat arsitektur model IndoBERT khusus klasifikasi urutan teks
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    ignore_mismatched_sizes=True # Mereset setelan kepala klasifikasi bawaan model
)

print("Model IndoBERT berhasil dimuat dan siap dilatih!")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model IndoBERT berhasil dimuat dan siap dilatih!


#####**Proses Training & Evaluasi Akhir**

#####Sel terakhir ini adalah inti dari praktikum Anda. Pertama, kita siapkan rumus perhitungan untuk mencari akurasi dan F1-score. Kedua, kita atur konfigurasi pelatihan (berjalan selama 1 Epoch sesuai instruksi tugas). Terakhir, perintah trainer.train() dijalankan untuk melatih model, diikuti oleh trainer.evaluate() untuk menguji performanya pada data uji yang belum pernah dilihat model sebelumnya.

In [9]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Membuat fungsi penghitung metrik performa model
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }

# 2. Mengatur konfigurasi parameter pelatihan (Poin 6)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,            # Menjalankan minimal 1 epoch sesuai instruksi tugas
    eval_strategy="epoch",         # <-- PARAMETER SUDAH DIPERBAIKI DI SINI
    logging_strategy="epoch"
)

# 3. Menyatukan model, argumen, data teks ter-token, dan rumus metrik ke dalam Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# 4. Mengeksekusi proses Fine-tuning (Poin 6)
print("\n=== [PROSES TRAINING DIMULAI] ===")
trainer.train()

# 5. Mengeksekusi pengujian performa akhir (Poin 7)
print("\n=== [HASIL EVALUASI AKHIR UNTUK LAPORAN] ===")
eval_results = trainer.evaluate()
print(f"Loss Akhir (Training Loss) : {eval_results['eval_loss']:.4f}")
print(f"Akurasi Model (Accuracy)   : {eval_results['eval_accuracy']:.4f}")
print(f"Nilai F1-Score             : {eval_results['eval_f1']:.4f}")


=== [PROSES TRAINING DIMULAI] ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.659929,0.669192,0.500000,0.485714


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



=== [HASIL EVALUASI AKHIR UNTUK LAPORAN] ===


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.659929,0.669192,1,0.500000,0.485714


Loss Akhir (Training Loss) : 0.6692
Akurasi Model (Accuracy)   : 0.5000
Nilai F1-Score             : 0.4857


#####**Kesimpulan**

#####Berdasarkan serangkaian proses praktikum fine-tuning model IndoBERT untuk klasifikasi teks ulasan, dapat ditarik beberapa kesimpulan utama sebagai berikut:





1.   Efektivitas Transfer Learning: Penggunaan model pre-trained IndoBERT base-p2 terbukti sangat efisien untuk klasifikasi teks bahasa Indonesia. Meskipun proses fine-tuning hanya dilakukan minimal 1 epoch dengan dataset yang relatif kecil (60 sampel), model mampu beradaptasi dan mengenali konteks sentimen dengan cepat karena sudah memiliki modal pengetahuan bahasa Indonesia yang kuat sebelumnya.
2.   Pentingnya Preprocessing yang Konsisten: Tahapan tokenisasi menggunakan AutoTokenizer dengan pengaturan padding dan truncation pada panjang maksimal 128 token berhasil menyamakan dimensi input teks. Hal ini memastikan struktur data teks kasual dapat dipetakan menjadi bentuk matriks numerik yang siap diproses oleh model tanpa kehilangan informasi penting.

3.   Analisis Performa Akhir: Proses pelatihan menghasilkan performa model yang diukur melalui metrik pada data uji sebesar:


*   Loss Akhir: [ 0.6692]
*   Akurasi (Accuracy): [0.5000]

*   F1-Score: [ 0.4857]
(Akurasi yang tinggi/bagus dipengaruhi oleh distribusi dataset yang seimbang antara ulasan positif dan negatif).
4.   Relevansi dalam Pengembangan Perangkat Lunak: Pendekatan fine-tuning berbasis Hugging Face Trainer API ini menunjukkan alur kerja (workflow) yang efisien dan modular, sehingga sangat potensial untuk diimplementasikan pada skala produksi yang lebih besar, seperti sistem analisis sentimen otomatis pada ulasan aplikasi atau platform e-commerce.











